In [3]:
include("../src/EBC.jl")
include("../../BeamToyModel/src/BeamToyModel.jl")

Main.BeamToyModel

$$
d^{\ell}_{m-1,n}(\beta) = \lambda _{m}a_{m-1}d^{\ell}_{m,n}(\beta) - \frac{a_{m-1}}{a_m}d^{\ell}_{m+1,n}(\beta) \\
 \lambda _{m} = \frac{n-m\cos{\beta}}{\sin{\beta}} \\
 a_{m} = \frac{2}{\sqrt{(\ell -m)(\ell+m+1)}}
$$

In [31]:
l = 100
n = 29
beta = pi/7

d_rec = wignerd_mdown_from_l(l, n, beta)
d_ref = [WignerD.wignerDjmn(l, m, n, 0.0, beta, 0.0) for m in 0:l]

maximum(abs.(d_rec .- d_ref))

34069.94561371786

In [32]:
d_ref

101-element Vector{ComplexF64}:
   -0.01458831105773083 + 0.0im
   -0.11332703283714635 + 0.0im
    -0.1314811768861632 + 0.0im
  -0.050719510021850656 + 0.0im
    0.07030822718329782 - 0.0im
    0.13271812964999613 - 0.0im
    0.07898868637458266 - 0.0im
    -0.0471387293053032 + 0.0im
   -0.12824509032105152 + 0.0im
   -0.08146848715837747 + 0.0im
                        ⋮
 -5.967704369677556e-13 + 0.0im
  9.364016035415137e-14 - 0.0im
 -1.574288522410297e-14 + 0.0im
  3.218497508612993e-15 - 0.0im
  6.712638642061317e-16 - 0.0im
 -5.211227467952999e-16 + 0.0im
 1.3065830889152193e-15 - 0.0im
  8.473543686630305e-16 - 0.0im
 1.2809368350517517e-15 - 0.0im

In [33]:
d_rec

101-element Vector{ComplexF64}:
       2486.050087409425 + 0.0im
      19312.494693574983 + 0.0im
       22406.21206917121 + 0.0im
       8643.306399500514 + 0.0im
     -11981.494886072662 + 0.0im
      -22617.00593793913 + 0.0im
     -13460.765258485917 + 0.0im
       8033.091812072185 + 0.0im
       21854.73813526426 + 0.0im
      13883.357629249414 + 0.0im
                         ⋮
   1.0169793231260633e-7 + 0.0im
  -1.6167319566644757e-8 + 0.0im
   2.3627234826830125e-9 - 0.0im
  -3.143146533968352e-10 + 0.0im
   3.754653574052615e-11 + 0.0im
 -3.9477322983257665e-12 - 0.0im
    3.53881294645787e-13 + 0.0im
 -2.5508705474505994e-14 + 0.0im
  1.2809368350517517e-15 - 0.0im

In [64]:
# ComplexF64 に ldexp をかける（2^e 倍）
@inline function cldexp(z::ComplexF64, e::Int)
    return ComplexF64(ldexp(real(z), e), ldexp(imag(z), e))
end

function wignerd_mdown_from_l_scaled(l::Int, n::Int, beta::Real;
    eps_sin::Real=1e-14,   # sinβ がこれ以下なら direct に逃げる
    beta_switch::Real=0.0, # 0 なら自動で決める
)
    (l >= 0) || throw(ArgumentError("l must be non-negative"))
    (abs(n) <= l) || throw(ArgumentError("n must satisfy |n| <= l"))

    dvals = zeros(ComplexF64, l + 1)

    # ---- 小β領域の分岐（sinβ判定だけだと足りないことがある）----
    β = float(beta)
    if beta_switch == 0.0
        # 雑だけど実務で効く：lが大きいほど小βを厳しく
        # （必要ならここは調整してOK）
        beta_switch = 1e-6 * (l + 1)
    end

    if abs(β) < beta_switch || abs(sin(β)) <= eps_sin
        for m in 0:l
            dvals[m + 1] = WignerD.wignerDjmn(l, m, n, 0.0, β, 0.0)
        end
        return dvals
    end

    # ---- 半角で sinβ, cosβ, λm を計算（キャンセル低減）----
    sh = sin(β/2)
    ch = cos(β/2)
    sinβ = 2*sh*ch
    cosβ = ch*ch - sh*sh

    # seed: d_{l,n}
    d_l_true = WignerD.wignerDjmn(l, l, n, 0.0, β, 0.0)
    dvals[l + 1] = d_l_true
    l == 0 && return dvals

    # ---- スケーリング状態：内部値 d_internal と指数 e（真値 = d_internal * 2^e）----
    e = 0

    # 内部状態（最初は真値をそのまま）
    d_m_plus1 = 0.0 + 0.0im      # d_{m+1}
    d_m       = d_l_true         # d_{m}

    # リスケ条件：conviqtっぽく「安全帯」に戻す（2の冪だけ使う）
    # ※値は適当に安全側。必要なら調整。
    BIG  = 2.0^450
    SMALL = 2.0^-450
    SHIFT = 200  # 一度に動かす2冪の指数

    @inline function rescale!(x::ComplexF64, y::ComplexF64, e::Int)
        # x,y を同じ2冪でまとめてスケール（線形同次なのでOK）
        ax = abs(x); ay = abs(y)
        amax = max(ax, ay)
        if amax == 0.0
            return x, y, e
        elseif amax > BIG
            # 2^SHIFT で割る
            x = cldexp(x, -SHIFT)
            y = cldexp(y, -SHIFT)
            e += SHIFT
        elseif amax < SMALL
            # 2^SHIFT で掛ける
            x = cldexp(x, SHIFT)
            y = cldexp(y, SHIFT)
            e -= SHIFT
        end
        return x, y, e
    end

    # 初回も念のため
    d_m, d_m_plus1, e = rescale!(d_m, d_m_plus1, e)

    for m in l:-1:1
        # a_{m-1}
        am1 = 2 / sqrt((l - (m - 1)) * (l + (m - 1) + 1))

        # λm を半角形で（n-m + 2m sh^2）/sinβ
        λm = ((n - m) + 2m * (sh*sh)) / sinβ

        d_m_minus1 = if m == l
            (λm * am1) * d_m
        else
            am = 2 / sqrt((l - m) * (l + m + 1))
            (λm * am1) * d_m - (am1 / am) * d_m_plus1
        end

        # 次ステップに備えてリスケ
        d_m_minus1, d_m, e = rescale!(d_m_minus1, d_m, e)

        # ここで「真値」を保存（ldexpで指数を反映）
        # dvals の添字は m-1 -> (m) だが、あなたの元コードに合わせて m を保存しているので踏襲
        dvals[m] = cldexp(d_m_minus1, e)

        # 状態更新（内部値のまま）
        d_m_plus1 = d_m
        d_m = d_m_minus1
    end

    # 最後に m=0 の要素を保存（ループ内で dvals[1] を入れていないため）
    dvals[1] = cldexp(d_m, e)

    return dvals
end

function wignerd_mdown_ratio_norm(l::Int, n::Int, beta::Real;
        eps_sin::Real=1e-14)

    (l >= 0) || throw(ArgumentError("l must be non-negative"))
    (abs(n) <= l) || throw(ArgumentError("n must satisfy |n| <= l"))

    β = float(beta)
    sh = sin(β/2)
    ch = cos(β/2)
    sinβ = 2*sh*ch
    cosβ = ch*ch - sh*sh

    dvals = zeros(Float64, l+1)

    # まず危険域は安全側の実装へ逃がす（β→0 の特異性を踏むので大事）
    if abs(sinβ) <= eps_sin
        for m in 0:l
            dvals[m+1] = real(WignerD.wignerDjmn(l, m, n, 0.0, β, 0.0))
        end
        return dvals
    end

    # --- 1) ratio r_m = d_{m+1}/d_m を下向きに計算 ---
    r = zeros(Float64, l+1)   # r[ m+1 ] = r_m
    r[l+1] = 0.0              # r_l = 0 （d_{l+1}=0）

    for m in l:-1:1
        am1 = 2 / sqrt((l-(m-1))*(l+(m-1)+1))
        # λm は半角で（キャンセルが減る）
        λm = ((n - m) + 2m*sh*sh) / sinβ
        A  = λm * am1

        C = if m == l
            0.0                 # d_{l+1}項は無いので C=0
        else
            am = 2 / sqrt((l-m)*(l+m+1))
            -(am1 / am)
        end

        denom = A + C*r[m+1]    # A + C r_m
        # denom が 0 に近いときはここが爆心地
        r[m] = 1.0 / denom      # r_{m-1}
    end

    # --- 2) 復元：d_l を種にして下へ ---
    #d_l = real(WignerD.wignerDjmn(l, l, n, 0.0, β, 0.0))
    d_l = seed_d_l_n_conviqt(l, n, beta)
    dvals[l+1] = d_l
    for m in l:-1:1
        # r[m] = d_m / d_{m-1} なので d_{m-1} = d_m / r[m]
        dvals[m] = dvals[m+1] / r[m]
    end

    # --- 3) 正規化：列（固定n）のノルムは 1 になるべき ---
    # 直交性：∑_m d_{m n}^2 = 1
    norm = sqrt(sum(abs2, dvals))
    dvals ./= norm

    # dvals[m+1] = d^l_{m,0}  (m=0..l) だと仮定
    #norm2_full = dvals[1]^2 + 2*sum(dvals[2:end].^2)
    #dvals ./= sqrt(norm2_full)

    # 位相（符号）を seed に合わせたいなら：
    # （正規化で全体符号が反転しても同じ解なので、好みで固定）
    if d_l != 0 && sign(dvals[l+1]) != sign(d_l)
        dvals .*= -1
    end

    return dvals
end

using SpecialFunctions

"""
conviqt式(27)(31)に対応する seed: d^l_{l,n}(β)
- logで絶対値を計算し、符号は (-1)^(l-n) として分離
- exp のオーバーフロー/アンダーフローを避けるため 2冪スケーリングで復元
"""
function seed_d_l_n_conviqt(l::Int, n::Int, beta::Real)
    (abs(n) <= l) || throw(ArgumentError("|n| must be <= l"))
    β = float(beta)

    sh = sin(β/2)
    ch = cos(β/2)

    # β→0 のとき sin(β/2)=0 で logが -Inf になるので、数学的にも多くが0
    # (l-n)>0 なら 0, (l-n)=0 なら (cos)^(2l)*A に落ちる
    if sh == 0.0
        if l == n
            # d^l_{l,l}(β)= (cos(β/2))^(2l) * A だが A=1
            return ch^(2l)
        else
            return 0.0
        end
    end

    # A = sqrt( (2l)! / ((l+n)!(l-n)!) )
    # logA = 0.5*(ln(2l)! - ln(l+n)! - ln(l-n)!)
    logA = 0.5 * (loggamma(2l + 1) - loggamma(l + n + 1) - loggamma(l - n + 1))

    # ln|d| = logA + (l+n)ln|cos| + (l-n)ln|sin|
    # (31)の形そのもの :contentReference[oaicite:2]{index=2}
    logabs = logA + (l + n) * log(abs(ch)) + (l - n) * log(abs(sh))

    # 符号：β∈(0,π) なら sin(β/2)>0 なので (-sin)^(l-n)=(-1)^(l-n)*sin^(l-n)
    sgn = isodd(l - n) ? -1.0 : 1.0

    # exp(logabs) を直接やると under/overflow するので 2冪で復元
    log2abs = logabs / log(2.0)
    e2 = floor(Int, log2abs)               # 2^e2 の指数
    mant = exp(logabs - e2 * log(2.0))     # mant ∈ (0,2) くらいに収まる
    return sgn * ldexp(mant, e2)
end

using SpecialFunctions

# d^l_{l,n}(β) を log で作る（倍率は正規化で消えるが、極小βでの安定性のため）
function seed_d_l_n_conviqt(l::Int, n::Int, beta::Real)
    (abs(n) <= l) || throw(ArgumentError("|n| <= l"))
    β = float(beta)
    sh = sin(β/2); ch = cos(β/2)

    if sh == 0.0
        return (l == n) ? ch^(2l) : 0.0
    end

    logA = 0.5 * (loggamma(2l + 1) - loggamma(l + n + 1) - loggamma(l - n + 1))
    logabs = logA + (l + n) * log(abs(ch)) + (l - n) * log(abs(sh))
    sgn = isodd(l - n) ? -1.0 : 1.0

    # 2冪で復元（overflow/underflow回避）
    log2abs = logabs / log(2.0)
    e2 = floor(Int, log2abs)
    mant = exp(logabs - e2 * log(2.0))
    return sgn * ldexp(mant, e2)
end

"""
conviqt式(7)の m0 方向再帰（ratio）で、固定 n に対し d^l_{m,n}(β) を m=0..l で返す。
- まず d^l_{n,m0}(β) を m0=l..0 で作り、
- 対称性 d_{mn}=(-1)^(m-n)d_{nm} で d^l_{m,n} に変換する
"""
function wignerd_mvec_fixedn_conviqt7(l::Int, n::Int, beta::Real; eps_sin::Real=1e-14)
    (l >= 0) || throw(ArgumentError("l>=0"))
    (abs(n) <= l) || throw(ArgumentError("|n|<=l"))

    β = float(beta)
    sh = sin(β/2); ch = cos(β/2)
    sinβ = 2*sh*ch
    cosβ = ch*ch - sh*sh

    # 返すのは m=0..l
    out = zeros(Float64, l+1)

    # βが極小なら別ルート（ここは好みで：あなたのWignerD.jl呼び出しでもOK）
    if abs(sinβ) <= eps_sin
        # β≈0 の極限は δ_{mn} なので m=|n|以外は0に近い
        # ただし厳密にやりたいなら外部関数でOK
        for m in 0:l
            out[m+1] = (m == n) ? 1.0 : 0.0
        end
        return out
    end

    mfix = n  # 式(7)の "m" を固定 = n
    # r[m0] = d_{mfix,m0+1} / d_{mfix,m0}
    r = zeros(Float64, l+1)
    r[l+1] = 0.0  # 境界：d_{mfix,l+1}=0 とみなせるので r_l=0

    # m0 = l,l-1,...,1 で r_{m0-1} を作る
    for m0 in l:-1:1
        t = (-mfix + m0*cosβ) / sinβ

        # α0 = 0.5*sqrt((l+m0)(l-m0+1))
        α0 = 0.5 * sqrt((l + m0) * (l - m0 + 1))
        # β0 = 0.5*sqrt((l-m0)(l+m0+1))
        β0 = 0.5 * sqrt((l - m0) * (l + m0 + 1))

        # 式(7)より：
        # α0 * d_{m0-1} = t*d_{m0} - β0*d_{m0+1}
        # なので q = d_{m0-1}/d_{m0} = (t - β0*r_m0)/α0
        q = (t - β0*r[m0+1]) / α0

        # r_{m0-1} = d_{m0}/d_{m0-1} = 1/q
        r[m0] = 1.0 / q
    end

    # 復元：seed として d_{mfix,l} を与えて、m0を下げる
    d_nm0 = zeros(Float64, l+1)   # d_nm0[m0+1] = d^l_{n,m0}
    d_nm0[l+1] = seed_d_l_n_conviqt(l, mfix, β)  # まず d^l_{l,mfix}
    # 欲しいのは d^l_{mfix,l} だが、d_{mfix,l} = (-1)^(mfix-l) d_{l,mfix}
    if isodd(mfix - l)
        d_nm0[l+1] *= -1
    end

    for m0 in l:-1:1
        # r[m0] = d_{m0}/d_{m0-1} なので d_{m0-1} = d_{m0}/r[m0]
        d_nm0[m0] = d_nm0[m0+1] / r[m0]
    end

    # ここで d_nm0[m+1] = d^l_{n,m}
    # 対称性で d^l_{m,n} に戻す：d_{m,n} = (-1)^(m-n) d_{n,m}
    for m in 0:l
        v = d_nm0[m+1]
        if isodd(m - n)
            v = -v
        end
        out[m+1] = v
    end

    # 正規化：もし con**viqt 側も「m=-l..l を含む列ノルム1」を使っているなら、
    # m>=0 しか持ってない out だけで norm を取ると一致しません。
    # まずは比較条件に合わせて、ここは "しない" か、同じ範囲で行ってください。
    # （とりあえず m=0..l だけでの正規化を入れるなら↓）
    # out ./= sqrt(sum(abs2,out))

    return out
end

wignerd_mvec_fixedn_conviqt7

In [65]:
l = 100
n = 2
beta = pi/7

d_rec = wignerd_mvec_fixedn_conviqt7(l, n, beta)
d_rec2 = wignerd_mdown_from_l_scaled(l, n, beta)
d_rec3 = wignerd_mdown_ratio_norm(l, n, beta)
d_ref = [WignerD.wignerDjmn(l, m, n, 0.0, beta, 0.0) for m in 0:l]

101-element Vector{ComplexF64}:
    -0.11247254326184039 + 0.0im
     0.04027859060932318 - 0.0im
     0.11451427528005581 - 0.0im
   -0.039245964484670166 + 0.0im
    -0.11328228689721687 + 0.0im
     0.04760332051253166 - 0.0im
      0.1078613042520364 - 0.0im
    -0.06451617411963682 + 0.0im
    -0.09515662920868082 + 0.0im
     0.08737832777113967 - 0.0im
                         ⋮
  -9.444062127196479e-16 + 0.0im
 -3.9093733815946877e-16 + 0.0im
  -9.537944045712684e-16 + 0.0im
  1.7004630659040784e-15 - 0.0im
   5.902600009614432e-16 - 0.0im
 -1.3813109157299874e-15 + 0.0im
  2.3353471223313823e-15 - 0.0im
   5.837104673484347e-16 - 0.0im
   3.467933590838601e-16 - 0.0im

In [66]:
maximum(abs.(d_rec3 .- d_ref))

0.07591282552383655

In [67]:
maximum(abs.(d_rec2 .- d_ref))

2.7840545633396834e19

In [71]:
# diagnostics for ratio method
norm_half_ref = sqrt(sum(abs2, d_ref))
d_ref_full = [WignerD.wignerDjmn(l, m, n, 0.0, beta, 0.0) for m in -l:l]
norm_full_ref = sqrt(sum(abs2, d_ref_full))

function ratio_denominators(l, n, beta)
    β = float(beta)
    sh = sin(β/2)
    sinβ = 2sh*cos(β/2)
    r = zeros(Float64, l+1)
    r[l+1] = 0.0
    denoms = zeros(Float64, l)
    for m in l:-1:1
        am1 = 2 / sqrt((l-(m-1))*(l+(m-1)+1))
        λm = ((n - m) + 2m*sh*sh) / sinβ
        A = λm * am1
        C = (m == l) ? 0.0 : -(am1 / (2 / sqrt((l-m)*(l+m+1))))
        denom = A + C*r[m+1]
        denoms[m] = denom
        r[m] = 1.0 / denom
    end
    return minimum(abs.(denoms)), maximum(abs.(r))
end

min_abs_denom, max_abs_ratio = ratio_denominators(l, n, beta)

(
    norm_half_ref = norm_half_ref,
    norm_full_ref = norm_full_ref,
    max_err_ratio = maximum(abs.(d_rec3 .- d_ref)),
    min_abs_denom = min_abs_denom,
    max_abs_ratio = max_abs_ratio,
    seed_diff = seed_d_l_n_conviqt(l, n, beta) - d_ref[end],
)

(norm_half_ref = 0.7206753542794911, norm_full_ref = 1.0000000000000002, max_err_ratio = 0.07591282552383655, min_abs_denom = 0.00693729324697645, max_abs_ratio = 144.14844009020953, seed_diff = -3.467933590838601e-16 + 0.0im)